# 🧠 Emotion Detector — Member 2 Training Notebook

**Pipeline:** Baseline → BiLSTM Tuning → DistilBERT Tuning → Evaluation → Visualizations

> **Before running:** Set runtime to **GPU** → Runtime → Change runtime type → T4 GPU

## 📁 Step 1: Mount Google Drive & Set Project Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ─── CHANGE THIS if your folder is in a different location ───
PROJECT_PATH = '/content/drive/MyDrive/Emotion_Detector'
# ─────────────────────────────────────────────────────────────

os.chdir(PROJECT_PATH)
print(f'✅ Working directory: {os.getcwd()}')
print(f'📂 Contents: {os.listdir()}')

## 📦 Step 2: Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install transformers --quiet
!pip install scikit-learn pandas numpy matplotlib seaborn tqdm --quiet
print('✅ All packages installed!')

## 🔍 Step 3: Verify Environment

In [ ]:
import torch
import sys
sys.path.insert(0, PROJECT_PATH)

print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU              : {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → GPU')

# Quick data check
from src.utils.data_loader import load_emotion_data
train, train_l, val, val_l, test, test_l = load_emotion_data()
print(f'\n✅ Data loaded: {len(train)} train | {len(val)} val | {len(test)} test')

---
## 🏃 Step 4: Train Baseline Model (TF-IDF + Logistic Regression)
*Fast — runs in ~30 seconds even on CPU*

In [ ]:
!python src/train_baseline.py

---
## ⚙️ Step 5: Hyperparameter Tuning — BiLSTM
*~10-15 minutes on GPU. Runs 9 experiments across 3 phases.*

In [ ]:
import sys
sys.path.insert(0, PROJECT_PATH)

from src.hyperparameter_tuning import tune_bilstm
best_bilstm_config = tune_bilstm()
print('\n🏆 Best BiLSTM config:', best_bilstm_config)

## 🔥 Step 6: Train Final BiLSTM (Best Hyperparameters)

In [ ]:
from src.train_deep import train_bilstm

# Use the best config found above, or set manually:
results = train_bilstm(
    lr=best_bilstm_config['lr'],
    batch_size=best_bilstm_config['batch_size'],
    dropout=best_bilstm_config['dropout'],
    num_epochs=25,
    patience=5,
    save_path='models/best_bilstm.pth',
    log_to_csv='experiments/hyperparameter_log.csv',
    experiment_name='bilstm_final_best'
)
print('✅ BiLSTM training complete!')

---
## ⚙️ Step 7: Hyperparameter Tuning — DistilBERT
*~15-25 minutes on GPU (T4). Runs 9 experiments.*

In [ ]:
from src.hyperparameter_tuning import tune_distilbert
best_bert_config = tune_distilbert()
print('\n🏆 Best DistilBERT config:', best_bert_config)

## 🤖 Step 8: Train Final DistilBERT (Best Hyperparameters)

In [ ]:
from src.train_distilbert import train_distilbert

results = train_distilbert(
    lr=best_bert_config['lr'],
    batch_size=best_bert_config['batch_size'],
    dropout=best_bert_config['dropout'],
    num_epochs=5,
    patience=3,
    save_path='models/best_distilbert.pth',
    log_to_csv='experiments/hyperparameter_log.csv',
    experiment_name='distilbert_final_best'
)
print('✅ DistilBERT fine-tuning complete!')

---
## 📊 Step 9: Evaluate All Models

In [ ]:
from src.evaluate_all import main as evaluate_all
evaluate_all()

## 📈 Step 10: Generate All Visualizations

In [ ]:
from src.visualize_results import main as visualize
visualize()

# Display the plots inline in Colab
from IPython.display import Image, display
import os

plots = [
    'results/training_curves.png',
    'results/model_comparison.png',
    'results/hyperparameter_sensitivity.png',
    'results/confusion_matrices.png',
    'results/per_class_f1.png',
]

for plot in plots:
    if os.path.exists(plot):
        print(f'\n📊 {plot}:')
        display(Image(plot))

---
## 📋 Step 11: View Hyperparameter Tuning Summary

In [ ]:
import pandas as pd
df = pd.read_csv('experiments/hyperparameter_log.csv')

print('=== HYPERPARAMETER LOG ===')
print(df.to_string(index=False))

print('\n=== BEST PER MODEL ===')
for model in df['model'].unique():
    best = df[df['model'] == model].loc[df[df['model'] == model]['val_acc'].idxmax()]
    print(f'{model}: val_acc={best["val_acc"]:.4f} | lr={best["lr"]} | batch={best["batch_size"]} | dropout={best["dropout"]}')

---
## ✅ Step 12: Deliverables Checklist

In [ ]:
import os

deliverables = [
    'models/best_model.pth',
    'models/best_bilstm.pth',
    'models/best_distilbert.pth',
    'results/predictions.csv',
    'results/metrics_summary.json',
    'experiments/hyperparameter_log.csv',
    'results/training_curves.png',
    'results/model_comparison.png',
    'results/confusion_matrices.png',
    'results/per_class_f1.png',
]

print('📦 DELIVERABLES FOR MEMBER 3')
print('=' * 55)
all_ok = True
for path in deliverables:
    exists = os.path.exists(path)
    if not exists:
        all_ok = False
    print(f'{"✅" if exists else "❌"} {path}')

print('\n' + ('🎉 All deliverables ready for Member 3!' if all_ok else '⚠️  Some files missing — check above steps.'))